# MODEL_BASELINE — NextWatch

Comparaison **dummy / baseline / 3 modeles**.

| Nom | Type | Idee |
| --- | --- | --- |
| **Dummy** | — | Films au hasard (plancher). |
| **Baseline** | heuristique | Meme genres + note moyenne. |
| **User–User CF** | **non supervise** | Voisins = fans du film A proches en **age/sexe** (pas d'historique user app). |
| **Film–Film CF** | **non supervise** | Voisins films (co-notes). |
| **Hybride Film×User** | **non supervise** | Film–Film enrichi par User–User (age/sexe). |

**Metriques :** Precision@5 / @10 (aime = note >= 4). **Seed :** 42.
Entrée produit = **age + sexe + film aime** uniquement.


## 1. Chargement et verification

Meme perimetre que l'EDA : exclusion `occupation` / `unknown`, genres en liste.


In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# --- Hyperparametres ---
RANDOM_STATE = 42
# Reproductibilite (split, echantillons, dummy).

LIKED_THRESHOLD = 4
# Seuil a partir duquel une note compte comme "film aime".

TRAIN_FRAC = 0.4  # au lieu de 0.50
MIN_SUPPORT = 10
CF_N_NEIGHBORS = 80  # au lieu de 40
HYBRID_ALPHA = 0.70  # au lieu de 0.65

np.random.seed(RANDOM_STATE)

GENRES = [
    "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western",
]

DATA_DIR = next(
    (p for p in [Path("dataset"), Path("../dataset")] if (p / "u.data.csv").exists()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("dataset/ introuvable")

notes = pd.read_csv(DATA_DIR / "u.data.csv")
films = pd.read_csv(DATA_DIR / "u.item.csv")
users = pd.read_csv(DATA_DIR / "u.user.csv", dtype={"zip_code": "string"}).drop(columns=["occupation"])

df_raw = notes.merge(users, on="user_id", how="left").merge(
    films, left_on="item_id", right_on="movie_id", how="left"
)
print(f"Genre unknown exclu : {int((df_raw['unknown']==1).sum())} notes")

df = df_raw.loc[df_raw["unknown"] != 1].drop(columns=["unknown"]).copy()
df["genres"] = df[GENRES].apply(lambda row: [g for g in GENRES if row[g] == 1], axis=1)

print(f"df : {len(df):,} notes | {df['user_id'].nunique()} users | {df['item_id'].nunique()} films")
display(df[["user_id", "item_id", "rating", "age", "gender", "title", "genres"]].head(3))


Genre unknown exclu : 10 notes
df : 99,990 notes | 943 users | 1680 films


,user_id,item_id,rating,age,gender,title,genres
0,196,242,3,49,M,Kolya (1996),[Comedy]
1,186,302,3,39,F,L.A. Confidential (1997),"[Crime, Film-Noir, Mystery, Thriller]"
2,22,377,1,25,M,Heavyweights (1994),"[Children's, Comedy]"


## 2. Split sans fuite

Split aleatoire 80/20 des notes. Stats et fit modeles **uniquement sur le train**.


In [11]:
train = df.sample(frac=TRAIN_FRAC, random_state=RANDOM_STATE)
test = df.drop(train.index)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
print(f"Train {len(train):,} | Test {len(test):,}")

movies = df.drop_duplicates("item_id").set_index("item_id")
movie_genres = movies["genres"].to_dict()
movie_title = movies["title"].to_dict()
# matrice genres 0/1 par film (catalogue)
film_genre_X = movies[GENRES].astype(float)
all_movie_ids = film_genre_X.index.to_numpy()
print(f"Films catalogue : {len(all_movie_ids)}")


Train 39,996 | Test 59,994
Films catalogue : 1680


## 3. Stats train

Note moyenne et volume de notes par film (baseline).


In [12]:
film_stats = (
    train.groupby("item_id")["rating"].agg(n_ratings="count", mean_rating="mean").reset_index()
)
eligible = film_stats.loc[film_stats["n_ratings"] >= MIN_SUPPORT].set_index("item_id")
mean_rating = eligible["mean_rating"]
print(f"Films eligible (>= {MIN_SUPPORT} notes) : {len(eligible)}")
display(eligible.head(3))


Films eligible (>= 10 notes) : 877


,n_ratings,mean_rating
item_id,,
1,175,3.868571
2,54,3.185185
3,36,3.166667


## 4. Cas d'evaluation

Requete A = meilleur film aime en train ; pertinents = aimes en test.
On conserve aussi la liste des films **deja aimes en train** pour les exclure des tops (evite de "reussir" en reproposant le passe).


In [13]:
liked_train = train.loc[train["rating"] >= LIKED_THRESHOLD]
liked_test = test.loc[test["rating"] >= LIKED_THRESHOLD]
train_counts = train.groupby("item_id").size()
cand_A = liked_train.copy()
cand_A["pop"] = cand_A["item_id"].map(train_counts)
idx_A = (
    cand_A.sort_values(["rating", "pop"], ascending=[False, False])
    .groupby("user_id", sort=False).first().reset_index()
)[["user_id", "item_id", "rating", "age", "gender"]]
idx_A = idx_A.rename(columns={"item_id": "query_id", "rating": "query_rating"})

liked_lists = liked_train.groupby("user_id")["item_id"].apply(set)
relevant = liked_test.groupby("user_id")["item_id"].apply(set).rename("relevant_ids").reset_index()
eval_cases = idx_A.merge(relevant, on="user_id", how="inner")
eval_cases["liked_train_ids"] = eval_cases["user_id"].map(liked_lists)
eval_cases["relevant_ids"] = eval_cases.apply(
    lambda r: r["relevant_ids"] - {r["query_id"]}, axis=1
)
eval_cases = eval_cases.loc[eval_cases["relevant_ids"].map(len) > 0].reset_index(drop=True)
print(f"Cas d'evaluation : {len(eval_cases)}")
print(f"Taille moyenne pertinents : {eval_cases['relevant_ids'].map(len).mean():.1f}")
display(eval_cases.head(3))


Cas d'evaluation : 937
Taille moyenne pertinents : 35.4


,user_id,query_id,query_rating,age,gender,relevant_ids,liked_train_ids
0,419,50,5,37,M,"{1, 514, 197, 134, 269, 14, 79, 275, 604, 100,...","{705, 257, 191, 69, 478, 488, 174, 50, 181, 28..."
1,549,50,5,42,M,"{1, 258, 515, 748, 237, 181}","{288, 866, 100, 50, 405, 118, 121, 127}"
2,296,50,5,43,F,"{256, 257, 258, 515, 1284, 7, 9, 523, 268, 14,...","{1, 514, 521, 11, 652, 269, 659, 20, 277, 276,..."


## 5. Entrainement des modeles (train uniquement)

### Film–Film CF (non supervise)
Similarite cosine film×film sur les notes train, voisins precomputes.

### User–User CF (non supervise, protocole **produit**)
En app on n'a **pas** l'historique MovieLens du user : seulement **age**, **sexe**, **film A**.
On prend les users train qui ont **aime A**, on les pondère par similarite age/sexe, puis on agregue leurs films aimes.

### Hybride Film×User
`HYBRID_ALPHA * FilmCF + (1 - HYBRID_ALPHA) * UserCF` (fusion de rangs).


In [14]:
# --- Profils users (age, sexe) pour User–User produit ---
user_profile = train.drop_duplicates("user_id").set_index("user_id")[["age", "gender"]].copy()

# --- Film–Film CF : matrice item × user + voisins ---
users_u = sorted(train["user_id"].unique())
items_u = sorted(train["item_id"].unique())
u_ix = {u: i for i, u in enumerate(users_u)}
i_ix = {it: i for i, it in enumerate(items_u)}
mat_ui = csr_matrix(
    (train["rating"].astype(float).values, (train["user_id"].map(u_ix).values, train["item_id"].map(i_ix).values)),
    shape=(len(users_u), len(items_u)),
)
mat_iu = mat_ui.T.tocsr()

nn_item = NearestNeighbors(metric="cosine", algorithm="brute")
nn_item.fit(mat_iu)
item_ids_arr = np.array(items_u)

CF_ITEM = {}
n_ask_i = min(CF_N_NEIGHBORS + 1, len(items_u))
d_i, ix_i = nn_item.kneighbors(mat_iu, n_neighbors=n_ask_i)
for i, mid in enumerate(items_u):
    CF_ITEM[mid] = [
        (int(item_ids_arr[j]), 1.0 - float(d))
        for d, j in zip(d_i[i], ix_i[i])
        if int(item_ids_arr[j]) != mid
    ]

# Notes aimees par user (panel train) — sert a agreger les voisins age/sexe
liked_by_user = (
    train.loc[train["rating"] >= LIKED_THRESHOLD]
    .groupby("user_id")[["item_id", "rating"]]
    .apply(lambda x: list(zip(x["item_id"], x["rating"])))
    .to_dict()
)

# Fans de chaque film (pour User–User produit : lookup rapide)
fans_of = {
    mid: list(zip(
        grp["user_id"].to_numpy(),
        grp["rating"].to_numpy(),
        grp["age"].to_numpy(),
        grp["gender"].to_numpy(),
    ))
    for mid, grp in train.loc[train["rating"] >= LIKED_THRESHOLD].groupby("item_id")
}

print(f"Film–Film CF : {len(CF_ITEM)} films (voisins cosine)")
print(f"User–User CF : voisins = fans(A) × similarite age/sexe (pas de user_id app)")
print(f"Hybride Film×User : alpha_film={HYBRID_ALPHA}")


Film–Film CF : 1538 films (voisins cosine)
User–User CF : voisins = fans(A) × similarite age/sexe (pas de user_id app)
Hybride Film×User : alpha_film=0.7


## 6. Recommenders

Regle commune : exclure le film requete et les films deja aimes en train.


In [15]:
def _seen(query_id, liked_train_ids):
    s = set(liked_train_ids) if liked_train_ids is not None else set()
    s.add(int(query_id))
    return s


def recommend_dummy(query_id, k=10, age=None, gender=None, liked_train_ids=None, user_id=None):
    seen = _seen(query_id, liked_train_ids)
    pool = np.array([m for m in all_movie_ids if m not in seen])
    local = np.random.default_rng(RANDOM_STATE + int(query_id))
    if len(pool) == 0:
        return []
    return local.choice(pool, size=min(k, len(pool)), replace=False).tolist()


def recommend_baseline(query_id, k=10, age=None, gender=None, liked_train_ids=None, user_id=None):
    seen = _seen(query_id, liked_train_ids)
    gq = set(movie_genres.get(query_id, []))
    if not gq:
        return []
    scores = [
        (mid, float(m))
        for mid, m in mean_rating.items()
        if mid not in seen and gq & set(movie_genres.get(mid, []))
    ]
    scores.sort(key=lambda x: (-x[1], x[0]))
    return [m for m, _ in scores[:k]]


def _demo_sim(age_a, gender_a, age_b, gender_b):
    sex = 1.0 if gender_a == gender_b else 0.0
    age_term = 1.0 - min(abs(float(age_a) - float(age_b)) / 50.0, 1.0)
    return 0.5 * sex + 0.5 * age_term


def recommend_user_cf(query_id, k=10, age=None, gender=None, liked_train_ids=None, user_id=None):
    """User–User CF protocole produit : uniquement age + sexe + film A.

    `user_id` est ignore (indisponible en app). Les voisins sont les users train
    qui ont aime A, ponderes par similarite demographique.
    """
    seen = _seen(query_id, liked_train_ids)
    age = 30.0 if age is None else float(age)
    gender = "M" if gender is None else gender

    neighbors = []
    for uid, rat, a, g in fans_of.get(int(query_id), []):
        demo = _demo_sim(age, gender, a, g)
        neighbors.append((int(uid), demo * float(rat) / 5.0))
    neighbors.sort(key=lambda x: -x[1])
    neighbors = neighbors[:CF_N_NEIGHBORS]

    scores = {}
    for vid, sim in neighbors:
        for mid, rat in liked_by_user.get(vid, []):
            if mid in seen:
                continue
            scores[mid] = scores.get(mid, 0.0) + float(sim) * float(rat)

    return [m for m, _ in sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:k]]


def recommend_film_cf(query_id, k=10, age=None, gender=None, liked_train_ids=None, user_id=None):
    seen = _seen(query_id, liked_train_ids)
    seeds = set(liked_train_ids) if liked_train_ids is not None else {query_id}
    seeds.add(int(query_id))
    seed_ratings = {}
    if user_id is not None:
        sub = train.loc[
            (train["user_id"] == user_id) & (train["rating"] >= LIKED_THRESHOLD),
            ["item_id", "rating"],
        ]
        seed_ratings = dict(zip(sub["item_id"], sub["rating"]))
    scores = {}
    for aid in seeds:
        w = float(seed_ratings.get(aid, LIKED_THRESHOLD))
        for mid, sim in CF_ITEM.get(aid, []):
            if mid in seen:
                continue
            scores[mid] = scores.get(mid, 0.0) + w * sim
    return [m for m, _ in sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:k]]


def recommend_hybrid(query_id, k=10, age=None, gender=None, liked_train_ids=None, user_id=None,
                      alpha=None, pool=80):
    """Film–Film CF enrichi / reranke par User–User CF (fusion de rangs)."""
    alpha = HYBRID_ALPHA if alpha is None else float(alpha)
    pool = max(pool, k)
    film_list = recommend_film_cf(
        query_id, k=pool, age=age, gender=gender,
        liked_train_ids=liked_train_ids, user_id=user_id,
    )
    user_list = recommend_user_cf(
        query_id, k=pool, age=age, gender=gender,
        liked_train_ids=liked_train_ids, user_id=user_id,
    )
    scores = {}
    for rank, mid in enumerate(film_list):
        scores[mid] = scores.get(mid, 0.0) + alpha / (1.0 + rank)
    for rank, mid in enumerate(user_list):
        scores[mid] = scores.get(mid, 0.0) + (1.0 - alpha) / (1.0 + rank)
    return [m for m, _ in sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:k]]


ex = eval_cases.iloc[0]
qid = int(ex["query_id"])
kw = dict(
    age=ex["age"], gender=ex["gender"],
    liked_train_ids=ex["liked_train_ids"], user_id=int(ex["user_id"]),
)
print("Query:", movie_title.get(qid))
print("Dummy   :", [movie_title.get(i) for i in recommend_dummy(qid, 5, **kw)])
print("Baseline:", [movie_title.get(i) for i in recommend_baseline(qid, 5, **kw)])
print("UserCF  :", [movie_title.get(i) for i in recommend_user_cf(qid, 5, **kw)])
print("FilmCF  :", [movie_title.get(i) for i in recommend_film_cf(qid, 5, **kw)])
print("Hybrid  :", [movie_title.get(i) for i in recommend_hybrid(qid, 5, **kw)])


Query: Star Wars (1977)
Dummy   : ['Die Hard 2 (1990)', '8 Heads in a Duffel Bag (1997)', 'Ninotchka (1939)', 'Judgment Night (1993)', 'Anaconda (1997)']
Baseline: ['Casablanca (1942)', "Schindler's List (1993)", 'Bridge on the River Kwai, The (1957)', 'Empire Strikes Back, The (1980)', 'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963)']
UserCF  : ['Toy Story (1995)', 'Godfather, The (1972)', 'Empire Strikes Back, The (1980)', 'Monty Python and the Holy Grail (1974)', 'Apollo 13 (1995)']
FilmCF  : ['Silence of the Lambs, The (1991)', 'Casablanca (1942)', 'Monty Python and the Holy Grail (1974)', 'Pulp Fiction (1994)', 'Shawshank Redemption, The (1994)']
Hybrid  : ['Silence of the Lambs, The (1991)', 'Toy Story (1995)', 'Casablanca (1942)', 'Monty Python and the Holy Grail (1974)', 'Pulp Fiction (1994)']


## 7. Evaluation Precision@5 et @10


In [16]:
def precision_at_k(recommended, relevant, k):
    top = recommended[:k]
    if not top:
        return 0.0
    return sum(1 for m in top if m in relevant) / k


def evaluate(recommend_fn, cases, ks=(5, 10), max_k=10):
    rows, cache = [], []
    for _, row in cases.iterrows():
        qid = int(row["query_id"])
        relevant = row["relevant_ids"]
        rec = recommend_fn(
            qid,
            k=max_k,
            age=row["age"],
            gender=row["gender"],
            liked_train_ids=row["liked_train_ids"],
            user_id=int(row["user_id"]),
        )
        cache.append({"user_id": row["user_id"], "query_id": qid, "rec": rec, "relevant": relevant})
        rows.append({f"P@{k}": precision_at_k(rec, relevant, k) for k in ks})
    return pd.DataFrame(rows).mean().to_frame("mean").T, cache


methods = [
    ("dummy_random", recommend_dummy),
    ("baseline_overlap_mean", recommend_baseline),
    ("model_user_user_cf", recommend_user_cf),
    ("model_film_film_cf", recommend_film_cf),
    ("model_hybrid_film_user", recommend_hybrid),
]

summaries, caches = [], {}
for name, fn in methods:
    s, c = evaluate(fn, eval_cases)
    summaries.append(s.assign(method=name))
    caches[name] = c

results = pd.concat(summaries, ignore_index=True)[["method", "P@5", "P@10"]]
print(f"Precision moyenne sur {len(eval_cases)} cas (seed={RANDOM_STATE})")
display(results.round(4))


Precision moyenne sur 937 cas (seed=42)


,method,P@5,P@10
0,dummy_random,0.0166,0.0172
1,baseline_overlap_mean,0.1878,0.1463
2,model_user_user_cf,0.3289,0.2853
3,model_film_film_cf,0.3471,0.3091
4,model_hybrid_film_user,0.3624,0.3185


## 8. Test situation produit (age + sexe + un film)

Meme metriques, mais **sans** `user_id` MovieLens.
Le User–User CF est **deja** calibre ainsi (age/sexe + film A) : eval §7 et produit §8 doivent coincider pour ce modele.
Le Film–Film / hybride peuvent encore utiliser l'historique train en eval via `liked_train_ids` (seeds) — a distinguer de l'entree app.


In [17]:
def evaluate_product(recommend_fn, cases, ks=(5, 10), max_k=10):
    rows = []
    for _, row in cases.iterrows():
        qid = int(row["query_id"])
        relevant = row["relevant_ids"]
        # pas de user_id : simulation produit
        rec = recommend_fn(
            qid,
            k=max_k,
            age=row["age"],
            gender=row["gender"],
            liked_train_ids=row["liked_train_ids"],
            user_id=None,
        )
        rows.append({f"P@{k}": precision_at_k(rec, relevant, k) for k in ks})
    return pd.DataFrame(rows).mean().to_frame("mean").T

prod_summaries = []
for name, fn in methods:
    s = evaluate_product(fn, eval_cases)
    summaries_prod.append(s.assign(method=name))

results_product = pd.concat(summaries_prod, ignore_index=True)[["method", "P@5", "P@10"]]
print(
    f"Situation produit (âge + sexe + 1 film) — Precision moyenne "
    f"sur {len(eval_cases)} cas (seed={RANDOM_STATE})"
)
display(results_product.round(4))

# Rappel section 7 (avec historique) pour comparer le CF
print("\nRappel section 7 (éval avec historique train, CF multi-seed) :")
display(results.round(4))


Situation produit (âge + sexe + 1 film) — Precision moyenne sur 937 cas (seed=42)


,method,P@5,P@10
0,dummy_random,0.0106,0.0096
1,baseline_overlap_mean,0.0754,0.0705
2,model_kmeans,0.0720,0.0585
3,model_item_cf,0.1427,0.1200
4,model_logistic,0.0592,0.0496
5,dummy_random,0.0166,0.0172
6,baseline_overlap_mean,0.1878,0.1463
7,model_user_user_cf,0.3289,0.2853
8,model_film_film_cf,0.3475,0.3098
9,model_hybrid_film_user,0.3658,0.3172



Rappel section 7 (éval avec historique train, CF multi-seed) :


,method,P@5,P@10
0,dummy_random,0.0166,0.0172
1,baseline_overlap_mean,0.1878,0.1463
2,model_user_user_cf,0.3289,0.2853
3,model_film_film_cf,0.3471,0.3091
4,model_hybrid_film_user,0.3624,0.3185


## 9. Analyse d'erreurs (baseline)

Trois cas ou la baseline a 0 hit@5.


In [18]:
cache_base = caches["baseline_overlap_mean"]
error_rows = [c for c in cache_base if not any(m in c["relevant"] for m in c["rec"][:5])]
print(f"Baseline 0 hit@5 : {len(error_rows)} / {len(cache_base)}")
error_rows = sorted(error_rows, key=lambda c: -len(c["relevant"]))[:3]

for i, c in enumerate(error_rows, 1):
    qid = c["query_id"]
    print("=" * 70)
    print(f"Erreur #{i} — user {c['user_id']}")
    print(f"  A : {movie_title.get(qid)} | genres={movie_genres.get(qid)} | cluster={film_cluster.get(qid)}")
    print("  Top5 baseline :")
    for mid in c["rec"][:5]:
        print(f"    - {movie_title.get(mid)} | mean={mean_rating.get(mid, float('nan')):.2f} | cl={film_cluster.get(mid)}")
    print(f"  Pertinents test (n={len(c['relevant'])}, echantillon) :")
    for mid in list(c["relevant"])[:5]:
        print(f"    - {movie_title.get(mid)} | genres={movie_genres.get(mid)} | cl={film_cluster.get(mid)}")


Baseline 0 hit@5 : 462 / 937
Erreur #1 — user 314
  A : Scream (1996) | genres=['Horror', 'Thriller'] | cluster=12
  Top5 baseline :
    - Some Folks Call It a Sling Blade (1993) | mean=4.50 | cl=15
    - Rear Window (1954) | mean=4.42 | cl=4
    - Close Shave, A (1995) | mean=4.41 | cl=5
    - Third Man, The (1949) | mean=4.40 | cl=4
    - Sling Blade (1996) | mean=4.32 | cl=15
  Pertinents test (n=100, echantillon) :
    - Casper (1995) | genres=['Adventure', "Children's"] | cl=10
    - Toy Story (1995) | genres=['Animation', "Children's", 'Comedy'] | cl=5
    - Men in Black (1997) | genres=['Action', 'Adventure', 'Comedy', 'Sci-Fi'] | cl=7
    - Twelve Monkeys (1995) | genres=['Drama', 'Sci-Fi'] | cl=7
    - Little Big League (1994) | genres=["Children's", 'Comedy'] | cl=10
Erreur #2 — user 286
  A : Indiana Jones and the Last Crusade (1989) | genres=['Action', 'Adventure'] | cluster=13
  Top5 baseline :
    - Treasure of the Sierra Madre, The (1948) | mean=4.24 | cl=1
    - Boot, D